# BaB on CIFAR-100 ResNet-medium — `conv_mode` A/B walkthrough

This notebook drives the same harness as `scripts/verify_resnet_cifar100.py` to exercise the ACT BaB + DualSolver stack on two specific VNN-COMP 2024 CIFAR-100 instances, under each of the three `conv_mode` configurations added in Waves 1–5:

| Instance | α-β-CROWN verdict | α-β-CROWN wall | Role in this notebook |
|---|---|---|---|
| `8028` | FALSIFIED | ~0.3s | Fast sanity check — exercises the CE-validation path under patches. |
| `3995` | CERTIFIED | ~43.5s | Perf target — the one we want ACT to eventually CERTIFY. |

### What each config tests
- **`matrix`** — baseline dense `LinearBound` A/b matrices (pre-W3 behaviour).
- **`patches` (mixed)** — `conv_mode=patches`, but `dispatch_conv_forward` is allowed to fall back to matrix with a `log.warning` when it receives a `LinearBound` input. This is today's default.
- **`patches_strict`** — same as above, but any fallback raises `RuntimeError`. Today this errors on the first conv because the input spec seeds `LinearBound`; once **Tier 2** (patches-aware input-spec seeding + BaBSR patches scoring) lands, this configuration should pass.

### How to use
Run cells top-to-bottom. The final DataFrame + bar chart makes it easy to diff before/after any of the Tier 1/2/3 fixes (see the concluding markdown for the checklist).

In [ ]:
import logging
import sys
from pathlib import Path

act_root = Path.cwd().resolve()
if not (act_root / "act").exists():
    act_root = act_root.parent
if str(act_root) not in sys.path:
    sys.path.insert(0, str(act_root))

from act.util.device_manager import initialize_device
initialize_device("cuda", "float32")

for noisy in (
    "act.front_end.vnnlib_loader",
    "act.pipeline.verification.act2torch",
    "act.back_end.counterexample_io",
):
    logging.getLogger(noisy).setLevel(logging.ERROR)

from scripts.verify_resnet_cifar100 import INSTANCE_REGISTRY, verify_instance

CONV_MODES = ["matrix", "patches", "patches_strict"]
COMMON_KWARGS = dict(
    device="cuda",
    subproblem_batch_size=16,
    eta_iters=10,
    max_depth=20,
    max_nodes=None,
    verbose=False,
)
print("registered instances:", list(INSTANCE_REGISTRY.keys()))

## 1. Sanity check — instance 8028 (FALSIFIED)

All three configs should return `FALSIFIED` in under a second because BaB finds a validated counterexample at the root. What differs across configs:
- `conv_mat` goes 0 → ~19 → 1 (patches_strict aborts on the first materialize, before the CE pipeline can finish).
- `warns` mirrors `conv_mat` — one warning per dispatch fallback.

Use this cell as a smoke test: if matrix mode here doesn't pass, something else is wrong before you start reading patches warnings.

In [ ]:
results_8028 = [
    verify_instance("8028", conv_mode_arg=mode, time_budget_s=30.0, **COMMON_KWARGS)
    for mode in CONV_MODES
]
for r in results_8028:
    tag = "OK  " if r.passed else "FAIL"
    strict = "+strict" if r.strict_patches else ""
    print(
        f"[{tag}] 8028 conv_mode={r.conv_mode}{strict:<7} status={r.status:<10} "
        f"wall={r.wall_time_s:5.2f}s nodes={r.nodes}  conv_mat={r.conv_materializations:<3} "
        f"warns={r.warning_count}"
    )

## 2. Perf target — instance 3995 (CERTIFIED by α-β-CROWN at 43.5s)

This is the benchmark we're trying to match. Today:
- `matrix` returns **UNKNOWN** at ~45s (nodes capped by budget; the primary gap is α quality, not memory layout — see Tier 3).
- `patches` (mixed) returns **UNKNOWN** at ~126s with ~342 conv materializations and a matching flood of warnings.
- `patches_strict` returns **ERROR** in <1s (first conv fallback trips the strict guard).

We cap each config at 60s of wall-clock budget so this cell runs in ~3–4 min total; raise `time_budget_s` if you want to measure closer-to-converged numbers.

In [ ]:
results_3995 = [
    verify_instance("3995", conv_mode_arg=mode, time_budget_s=60.0, **COMMON_KWARGS)
    for mode in CONV_MODES
]
for r in results_3995:
    tag = "OK  " if r.passed else "FAIL"
    strict = "+strict" if r.strict_patches else ""
    print(
        f"[{tag}] 3995 conv_mode={r.conv_mode}{strict:<7} status={r.status:<10} "
        f"wall={r.wall_time_s:5.2f}s nodes={r.nodes}  conv_mat={r.conv_materializations:<4} "
        f"warns={r.warning_count}"
    )

## 3. Side-by-side comparison

Two views:
1. A pandas DataFrame with all numeric results (and α-β-CROWN reference where available).
2. A matplotlib bar chart of wall-time per (instance, config), with the α-β-CROWN baseline drawn as a horizontal dashed line.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for r in results_8028 + results_3995:
    rows.append(
        {
            "instance": r.instance,
            "conv_mode": f"{r.conv_mode}{'+strict' if r.strict_patches else ''}",
            "status": r.status,
            "wall_s": round(r.wall_time_s, 2),
            "nodes": r.nodes,
            "peak_vram_gb": None if r.peak_vram_gb is None else round(r.peak_vram_gb, 2),
            "conv_mat": r.conv_materializations,
            "warns": r.warning_count,
            "abcrown_status": r.abcrown_status,
            "abcrown_wall_s": r.abcrown_wall_s,
            "passed": r.passed,
        }
    )
df = pd.DataFrame(rows)
display(df)

fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"{r['instance']}\n{r['conv_mode']}" for r in rows]
colors = ["tab:green" if r["passed"] else "tab:red" for r in rows]
ax.bar(labels, [r["wall_s"] for r in rows], color=colors)
for inst in ("8028", "3995"):
    ref = INSTANCE_REGISTRY[inst]["abcrown_wall_s"]
    ax.axhline(ref, linestyle="--", alpha=0.4, color="black")
    ax.text(len(labels) - 0.3, ref, f" α-β-CROWN {inst}: {ref}s", va="bottom", fontsize=8)
ax.set_ylabel("wall time (s)")
ax.set_title("ACT verify_bab wall-time per (instance, conv_mode)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## 4. Interpretation & what to fix next

Read the tables above against this checklist. After each fix, re-run cells 2 and 3 and diff the DataFrame.

### Tier 1 — Merge prep (~8–12 h)
Mostly docs + Oracle review + minor log throttling. This notebook's output **should not change** after Tier 1 — any delta indicates an accidental behavioural change in what was supposed to be a docs-only wave.

### Tier 2 — Make patches actually propagate (~2–3 d)
After Tier 2 you expect:
- `patches_strict` row for instance 3995 flips **ERROR → UNKNOWN (or CERTIFIED)** — no more fallback.
- `conv_mat` drops to 0 on both patches rows.
- `peak_vram_gb` for patches drops from ~83 GB to single-digit GB at batch=16 (exact number depends on the network size; stretch is ≤ 30 GB at batch=256).
- `warns` on `patches` row drops to 0 (BaBSR no longer densifies either).
- Matrix row is **unchanged**.

### Tier 3 — Actually hit CERTIFIED on 3995 (~1–2 wk)
Primarily independent α-intermediate/α-final (plan §9 #6, ~300 LOC). After Tier 3 you expect:
- Instance 3995 `status` flips **UNKNOWN → CERTIFIED** on `matrix`, `patches`, and `patches_strict` rows.
- `wall_s` for 3995 falls below the α-β-CROWN line (~43.5s).
- Patches savings enable `batch=256` runs without OOM.

### Known-good invariants (must hold at every tier)
- Instance 8028 stays FALSIFIED across all three configs (soundness).
- Instance 3995 never returns FALSIFIED on any config (soundness — α-β-CROWN certifies it; any FALSIFIED here is a sound-bound bug).
- `passed=False` on any matrix row indicates a regression in the baseline path.